In [1]:
import numpy as np
import pylab as py
import matplotlib.patches as mpatches
import os
## This is the base folder
BASE='/ga/amit/BIO/FINAL'
os.chdir(BASE)
import sys
sys.path.append("CODE")
print(sys.path)
from skimage import io, morphology
from utils import augment_background_thick, color_image, load_model,  get_file_numbers
from analyze_to_csv import process_files, analyze_p, match_points_from_ims, analyze_cell
import torch
import pandas as pd
from PIL import Image
from predict import predict, predict_file
import re
import os
import shutil
gn='0'
device = torch.device("cuda:"+gn if torch.cuda.is_available() else "cpu")
data_path='data/'
%load_ext autoreload
%autoreload 2

['/ga/amit/BIO/FINAL', '/opt/anaconda3_new/lib/python312.zip', '/opt/anaconda3_new/lib/python3.12', '/opt/anaconda3_new/lib/python3.12/lib-dynload', '', '/home/amit/.local/lib/python3.12/site-packages', '/opt/anaconda3_new/lib/python3.12/site-packages', '/opt/anaconda3_new/lib/python3.12/site-packages/setuptools/_vendor', 'CODE']


### The two models we want to use: actin$\rightarrow$junction, pred_junction$\rightarrow$outline

In [5]:
model_a="actin_junction_mix_0.1_kernel_5_nlayers_4_ds_100_lrstep_100_ws_200_fl_0_1"
model_o="junction_outline_mix_0.0_kernel_5_nlayers_4_ds_0_lrstep_100_ws_200_fl_0_bdy_10_mrg_40_1"
#model_o="pred_junction_outline_mix_0.0_kernel_5_nlayers_4_ds_0_lrstep_100_ws_200_fl_0_bdy_10_mrg_40_1"
model_l=None

In [6]:
target='test/'
datapath='./data/'
dfp,iml,iml_p=analyze_p(device,model_a,model_o, model_name_l=model_l, reduced=1, datapath=datapath)
dfpgt,_,_=analyze_p(device,model_a,model_o, model_name_l=model_l , reduced=1, gt=True, dfp=dfp,datapath=datapath)
dfpgt.to_excel('junction_outline_prediction_comparison.xlsx')

actin_junction_mix_0.1_kernel_5_nlayers_4_ds_100_lrstep_100_ws_200_fl_0_1 junction_outline_mix_0.0_kernel_5_nlayers_4_ds_0_lrstep_100_ws_200_fl_0_bdy_10_mrg_40_1 None
[  1   3  22  23  24  30  31  35  36  37  41  44  45  48  53  62  80  83
  84  89  93 100 103 104 109 111 112 113 114 121 123 125 126 132 133 135
 138 139 142 148 150 151 156 159 163 165 168 169 192 196 198 202 208 213
 215 219 220 222 236 237]
1 ['UF_outline1.tif', 'UF_junction1.tif', 'UF_actin1.tif']
actin_junction_mix_0.1_kernel_5_nlayers_4_ds_100_lrstep_100_ws_200_fl_0_1 junction_outline_mix_0.0_kernel_5_nlayers_4_ds_0_lrstep_100_ws_200_fl_0_bdy_10_mrg_40_1 None
reduced 1
data 36
3 ['UF_outline3.tif', 'UF_junction3.tif', 'UF_actin3.tif']
actin_junction_mix_0.1_kernel_5_nlayers_4_ds_100_lrstep_100_ws_200_fl_0_1 junction_outline_mix_0.0_kernel_5_nlayers_4_ds_0_lrstep_100_ws_200_fl_0_bdy_10_mrg_40_1 None
reduced 1
data 35
22 ['UF_actin22.tif', 'UF_junction22.tif', 'UF_outline22.tif']
actin_junction_mix_0.1_kernel_5_nlaye

### Create folders with the predicted junctions and predicted outline images. This is not essential and can be skipped for the following cells.

In [ ]:
predict(device,model_a,None,x_prefix='actin',y_prefix='junction')
predict(device,model_o,model_a,x_prefix='pred_junction',y_prefix='outline')

In [3]:
model_a="actin_junction_mix_0.1_kernel_5_nlayers_4_ds_100_lrstep_100_ws_200_fl_0_1"
model_o="pred_junction_outline_mix_0.0_kernel_5_nlayers_4_ds_0_lrstep_100_ws_200_fl_0_bdy_10_mrg_40_1"
model_l="junction_leakiness_mix_1.0_ws_200_zero_weight_0.4_leak_thresh_0.1_fl_0_a_1.0_1"


### Analyze cells in predicted outline images and write to an excel file. Then add the information from the ground truth outline images.

In [ ]:
target='test/'
datapath='./data/permeability/'
dfp,iml,iml_p=analyze_p(device,model_a,model_o, model_name_l=model_l, reduced=1, datapath='data/permeability/')
dfpgt,_,_=analyze_p(device,model_a,model_o, model_name_l=model_l , reduced=1, gt=True, dfp=dfp,datapath='data/permeability/')
dfpgt.to_excel(model_l+'.xlsx')

#### Prepare a legend for boundary types

In [ ]:
def get_legend_patches(reduce=False):
    
    colors = ['red', 'yellow','green']
    legend_patches = []
    if not reduce:
        labels = ['Junction', 'Thick', 'Broken']
        for color, label in zip(colors, labels):
            patch = mpatches.Patch(color=color, label=label)
            legend_patches.append(patch)
    else:
        colors = ['red','green']
        labels = ['Junction', 'Broken']
        for color, label in zip(colors,labels):
            patch = mpatches.Patch(color=color, label=label)
            legend_patches.append(patch)
    return legend_patches



### Show predicted and true outlines and connect matching cells. Also show predicted and true junctions and original actin image.



In [ ]:


def show_outline_new(k,ax,model_a,model_o,target,i,reduce=False,data_path='data/'):

    ima, imj, imo, iml, imj_p, imo_p, iml_p, celltype=process_files(device,i,model_a, model_o,target,datapath=data_path)  
    numim=4
    centt, centt_m, ctp, centp, JJ, cdp ,cdt = match_points_from_ims(i,imo_p, imo, ima, imj, celltype)
 
    imbb=np.concatenate((imo_p,imo),axis=1)

    imtaa=augment_background_thick(imo,10,40)
    
    ll=len(np.unique(imo_p))
    llt=len(np.unique(imo))
   
    imn=0
    
    ax[k,imn].imshow(color_image(imbb,reduce=reduce))
    ax[k,imn].plot([1053,1053],[0,1500],linewidth=2,color='black')
    ax[k,imn].set_title('Predicted outline          ground truth',fontsize=20)
    if centt is not None:
        ax[k,imn].scatter(centt[:,1]+1053,centt[:,0],s=100,color='blue')
        if centp is not None:
            ax[k,imn].scatter(centp[:,1],centp[:,0],s=100,color='black')
        if len(ctp)>0:
            ax[k,imn].scatter(ctp[:,1],ctp[:,0],s=100,color='blue')
            for ii in range(len(centt_m)):
                 ax[k,imn].plot([centt_m[ii,1]+1053,ctp[ii,1]],[centt_m[ii,0],ctp[ii,0]],linewidth=4,color='orange')
    
    ax[k,imn].xaxis.set_visible(False)  # Hide X-axis
    ax[k,imn].yaxis.set_visible(False)
    imn+=1
        
    

    ax[k,imn].imshow(imj_p)
    ax[k,imn].axis('off')
    ax[k,imn].set_title('Predicted \n junctions',fontsize=20)

    imn+=1
    ax[k,imn].imshow(imj)
    ax[k,imn].axis('off')
    ax[k,imn].set_title('Junctions',fontsize=20)

    imn+=1
    ax[k,imn].imshow(ima)
    ax[k,imn].axis('off')
    ax[k,imn].set_title('Actin',fontsize=20)
    return JJ, cdp, cdt

In [ ]:
model_a="actin_junction_mix_0.1_kernel_5_nlayers_4_ds_100_lrstep_100_ws_200_fl_0_1"
model_o="pred_junction_outline_mix_0.0_kernel_5_nlayers_4_ds_0_lrstep_100_ws_200_fl_0_bdy_10_mrg_40_1"
model_l=None

### Show some predictions of outlines and compare to ground truth.

In [ ]:
reduce=True
target='test/'
data_path='data/'
ii=get_file_numbers(data_path+target)
ii=np.unique(ii)
print(len(ii),'images')
np.random.shuffle(ii)

nim=10 # Number of input images to process
first=20 # First input image to process
first=np.minimum(len(ii)-nim,first)
print('first image',first,'number of images',nim,'reduced',reduce)
legend_patches=get_legend_patches(reduce=reduce)

fig, ax=py.subplots(nim,4,figsize=(20,8*nim),gridspec_kw={'width_ratios': [2,1,1,1]})
ax=ax.reshape(nim,4)

fig.tight_layout()
#ii=np.sort(ii)

for k,i in enumerate(ii[first:first+nim]):

    JJ, cdp, cdt= show_outline_new(k,ax,model_a,model_o,target,i,reduce=reduce,data_path=data_path)

ax[0,0].legend(handles=legend_patches,bbox_to_anchor=(0., 1.3, .5, 0),fontsize=16)

py.savefig('outline_matches.png')

### Show leakiness and outlines predicted from junctions.

In [ ]:
def keep_only_outline(ax):
        ax.xaxis.set_visible(False)  # Hide X-axis
        ax.yaxis.set_visible(False)

def compare_leakiness_outlines(ax,numfigs,i,zero_thresh,leak_thresh,model_name_leak,model_name_outline,fig,
                               model_name_junction=None,tt='test/',reduce=False, data_path='data/',mask=False, mix=False):
    py.rcParams.update({'font.size': 7})

    dir='data/permeability/'
    leak_name='leakiness'
    x_prefix='junction'
    pd=40
    
    if 'pred' in model_name_leak:
        model_junction=model_name_junction
    else:
        model_junction=None
    # Predict leakiness either from junction or from predicted junction. 
    #If the latter first get the predicted junction from the actin
    if 'junction' in model_name_leak:
        im_leak_pred,_=predict_file(device,tt, model_name_leak, i, model_junction_name=model_junction,
                     x_prefix='junction', y_prefix='leakiness', pad_size=pd, zero_thresh=zero_thresh,data_path=data_path)
    # Otherwise get leakiness from actin directly
    else:
        im_leak_pred,_=predict_file(device,tt, model_name_leak, i, model_junction_name=model_junction,
                     x_prefix='actin', y_prefix='leakiness', pad_size=pd, zero_thresh=zero_thresh,data_path=data_path)
    
    # Binarize the leakiness prediction
    im_leak_pred[im_leak_pred>0]=1
    nump=7
   
    imn=0

    if 'pred' in model_name_outline:
        model_junction=model_name_junction
    else:
        model_junction=None
    # Predict outline, either directly from junction or from predicted junction.
    #If the latter first get the predicted junction from the actin
    im_out_pred, im_junc_pred=predict_file(device,tt, model_name_outline, i, model_junction_name=model_name_junction,
                     x_prefix='junction', y_prefix='outline', pad_size=pd, zero_thresh=zero_thresh,data_path=data_path)
    im_outline_pred=color_image(im_out_pred,reduce=reduce)


    #ax=fig.add_subplot(numfigs,nump,imn)
    ax[i-1,imn].imshow(im_outline_pred) 
    keep_only_outline(ax[i-1,imn])
    
    ax[i-1,imn].set_title('Outlines')
    imn+=1

    if i==1:
        legend_patches=get_legend_patches(reduce=reduce)
        ax[i-1,1].legend(handles=legend_patches,bbox_to_anchor=(1.1, 1.7, 0.3, 0.),fontsize=16)  
   
    # If mask, only show and compare results on predicted boundaries.
    if mask:
        maskim=(im_out_pred>3) 
        im_leak_pred[maskim]=0
    
    # Get linear and thick prediction
    bdy=np.logical_or(im_out_pred==2,im_out_pred==1)
    # Get broken prediction
    broken=(im_out_pred==3)
    
    im_pred_bdy=np.zeros((im_leak_pred.shape[0],im_leak_pred.shape[1],4))
    im_pred_bdy[bdy,0]=1
    im_pred_bdy[bdy,3]=.2
    im_pred_broken=np.zeros((im_leak_pred.shape[0],im_leak_pred.shape[1],4))
    im_pred_broken[broken,1]=1
    im_pred_broken[broken,3]=.5

    pl_bdy=np.sum(im_leak_pred[bdy])/np.sum(bdy)
    pl_broken=np.sum(im_leak_pred[broken])/np.sum(broken)
    
    #fig.add_subplot(numfigs,nump,imn)
    ax[i-1,imn].imshow((1-im_leak_pred),cmap='gray',alpha=.5)
    ax[i-1,imn].imshow(im_pred_bdy)
    ax[i-1,imn].set_title(f'Leakiness prediction \n on predicted linear+thick \n Ratio {pl_bdy: .3f}')
    keep_only_outline(ax[i-1,imn])
    imn+=1

    
    #fig.add_subplot(numfigs,nump,imn)
    ax[i-1,imn].imshow((1-im_leak_pred)*.5,cmap='gray',alpha=.5)
    ax[i-1,imn].imshow(im_pred_broken)
    ax[i-1,imn].set_title(f'Leakiness prediction \n on predicted broken \n Ratio {pl_broken: .3f}')
    keep_only_outline(ax[i-1,imn])
    imn+=1

    
    im_leak_true=py.imread(dir+tt+leak_name+str(i)+'.tif')
    im_leak_true=im_leak_true/255.
    # Binarize the leakiness
    im_leak_true=(im_leak_true>leak_thresh)

    if mask:
        im_leak_true[maskim]=0

    S11=np.sum(np.logical_and(im_leak_true, im_leak_pred))
    T1=(np.sum(im_leak_true))
    C11=S11/T1

    S00=np.sum(np.logical_and(1-maskim,np.logical_and(1-im_leak_true, 1-im_leak_pred)))
    T0=(np.sum(np.logical_and(1-maskim,1-im_leak_true)))
    C00=S00/T0
    #print(C11,C00)
    l_bdy=np.sum(im_leak_true[bdy])/np.sum(bdy)
    l_broken=np.sum(im_leak_true[broken])/np.sum(broken)
    
    imtt=np.copy(im_leak_true)
    imtt[imtt<leak_thresh]=0
    #fig.add_subplot(numfigs,nump,imn)
    loss=np.mean((imtt-im_leak_pred)**2)
    ax[i-1,imn].imshow(1.-imtt,cmap='gray',alpha=.5)
    ax[i-1,imn].imshow(im_pred_bdy)
    pl=(f'leakiness with \n predicted linear+thick\n Ratio {l_bdy:,.3f}')
    ax[i-1,imn].set_title(pl)
    keep_only_outline(ax[i-1,imn])
    imn=imn+1

   
    imtt=np.copy(im_leak_true)
    imtt[imtt<leak_thresh]=0
    #fig.add_subplot(numfigs,nump,imn)
    loss=np.mean((imtt-im_leak_pred)**2)
    ax[i-1,imn].imshow(1.-imtt,cmap='gray',alpha=.5)
    ax[i-1,imn].imshow(im_pred_broken)
    
    pl=(f'leakiness with \n predicted broken \n Ratio {l_broken:,.3f} ')
    ax[i-1,imn].set_title(pl)
    keep_only_outline(ax[i-1,imn])
    imn=imn+1

    # Show junction image
 
    junction_file=dir+tt+'/'+x_prefix+str(i)+'.tif'
    imja=py.imread(junction_file)
    img = Image.fromarray(imja)
    #fig.add_subplot(numfigs,nump,imn)
    ax[i-1,imn].imshow(imja,cmap='gray')
    ax[i-1,imn].set_title('Junction')
    keep_only_outline(ax[i-1,imn])
    imn=imn+1
    # Show predicted junction image
    #fig.add_subplot(numfigs,nump,imn)
    ax[i-1,imn].imshow(im_junc_pred,cmap='gray')
    ax[i-1,imn].set_title('Predicted \n Junction')
    keep_only_outline(ax[i-1,imn])
    return S00, T0, S11, T1
    

### Run the comparison on 10 test images. Direct actin to leakiness model.

In [ ]:
model_a="actin_junction_mix_0.1_kernel_5_nlayers_4_ds_100_lrstep_100_ws_200_fl_0_1"
model_o="pred_junction_outline_mix_0.0_kernel_5_nlayers_4_ds_0_lrstep_100_ws_200_fl_0_bdy_10_mrg_40_1"
model_l="actin_leakiness_mix_1.0_ws_200_zero_weight_0.4_leak_thresh_0.1_fl_0_a_1.0_0"

tt='test/'
data_path='data/permeability/'
model_name_leak=model_l
model_name_outline=model_o
reduce=True
zero_thresh=0.8
leak_thresh=0.2
#fig=py.figure(figsize=(16,48))
nim=10
fig, ax=py.subplots(nim,7,figsize=(16,4*nim),layout='constrained')
ax=ax.reshape(nim,7)
OUT=[]
for i in range(nim):
    out=compare_leakiness_outlines(ax,10,i+1,zero_thresh,leak_thresh,model_name_leak,model_name_outline,fig,
                               model_name_junction=model_a,reduce=reduce,data_path=data_path,mask=True)
    OUT+=[out]
SS=np.sum(np.array(OUT),axis=0)
fig.suptitle(f'Actin to Leakiness correct rates:\n Non-leak: {SS[0]/SS[1]:,.3f}, leak: {SS[2]/SS[3]:,.3f},ACC: {(SS[0]+SS[2])/(SS[1]+SS[3]):,.3f}',fontsize=30)
py.savefig('Actin-leak.png')
print('Correct rates','Non-leak',SS[0]/SS[1],'leak',SS[2]/SS[3],'ACC',(SS[0]+SS[2])/(SS[1]+SS[3]) )

### Run the comparison on 10 test images. Default junction to leakiness model.

In [ ]:
model_a="actin_junction_mix_0.1_kernel_5_nlayers_4_ds_100_lrstep_100_ws_200_fl_0_1"
model_o="pred_junction_outline_mix_0.0_kernel_5_nlayers_4_ds_0_lrstep_100_ws_200_fl_0_bdy_10_mrg_40_1"
model_l="junction_leakiness_mix_1.0_ws_200_zero_weight_0.4_leak_thresh_0.1_fl_0_a_1.0_1"

tt='test/'
data_path='data/permeability/'
model_name_leak=model_l
model_name_outline=model_o
reduce=True
zero_thresh=0.8
leak_thresh=0.2
nim=10
fig, ax=py.subplots(nim,7,figsize=(16,4*nim),layout='constrained')
ax=ax.reshape(nim,7)
OUT=[]
for i in range(10):
    out=compare_leakiness_outlines(ax,10,i+1,zero_thresh,leak_thresh,model_name_leak,model_name_outline,fig,
                               model_name_junction=model_a,reduce=reduce,data_path=data_path,mask=True)
    OUT+=[out]
SS=np.sum(np.array(OUT),axis=0)
fig.suptitle(f'Junction to Leakiness correct rates:\n Non-leak: {SS[0]/SS[1]:,.3f}, leak: {SS[2]/SS[3]:,.3f},ACC: {(SS[0]+SS[2])/(SS[1]+SS[3]):,.3f}',fontsize=30)
py.savefig('Junction-leak.png')
print('Total correct rates','Non-leak',SS[0]/SS[1],'leak',SS[2]/SS[3],'ACC',(SS[0]+SS[2])/(SS[1]+SS[3]))

### Run the comparison on 10 test images. Predicted Junction to leakiness model. 

In [ ]:
model_a="actin_junction_mix_0.1_kernel_5_nlayers_4_ds_100_lrstep_100_ws_200_fl_0_1"
model_o="pred_junction_outline_mix_0.0_kernel_5_nlayers_4_ds_0_lrstep_100_ws_200_fl_0_bdy_10_mrg_40_1"
model_l="pred_junction_leakiness_mix_1.0_ws_200_zero_weight_0.4_leak_thresh_0.1_fl_0_a_1.0_1"

tt='test/'
data_path='data/permeability/'
model_name_leak=model_l
model_name_outline=model_o
reduce=True
zero_thresh=0.8
leak_thresh=0.2
nim=10
fig, ax=py.subplots(nim,7,figsize=(16,4*nim),layout='constrained')
ax=ax.reshape(nim,7)
#fig=py.figure(figsize=(16,48))
OUT=[]
for i in range(10):
    out=compare_leakiness_outlines(ax,10,i+1,zero_thresh,leak_thresh,model_name_leak,model_name_outline,fig,
                               model_name_junction=model_a,reduce=reduce,data_path=data_path,mask=True)
    OUT+=[out]
SS=np.sum(np.array(OUT),axis=0)
fig.suptitle(f'Pred-Junction to Leakiness correct rates:\n Non-leak: {SS[0]/SS[1]:,.3f}, leak: {SS[2]/SS[3]:,.3f},ACC: {(SS[0]+SS[2])/(SS[1]+SS[3]):,.3f}',fontsize=30)
py.savefig('Predjunction-leak.png')
print('Total correct rates','Non-leak',SS[0]/SS[1],'leak',SS[2]/SS[3],'ACC',(SS[0]+SS[2])/(SS[1]+SS[3]))
